## 1. Subscribe to the model package

To subscribe to the model package:
1. Open the model package listing page of **Medical Decision Model - Small** on AWS Marketplace.
1. On the AWS Marketplace listing, click on the **Continue to subscribe** button.
1. On the **Subscribe to this software** page, review and click on **"Accept Offer"** if you and your organization agrees with EULA, pricing, and support terms.
1. Once you click on **Continue to configuration button** and then choose a **region**, you will see a **Product Arn** displayed. This is the model package ARN that you need to specify while creating a deployable model using Boto3. Copy the ARN corresponding to your region and specify the same in the following cell.

- **Model**: `JSL-Medical-DM-Small`
- **Model Description**: Medical Decision Model - Small: typed questions about a medical record, and the guards PHI Guard v0.7 and PHI Guard v0.7 Fast

In [1]:
model_package_arn = "<Customer to specify Model package ARN corresponding to their AWS region>"

In [ ]:
import json
from urllib.parse import urlparse

import boto3
import pandas as pd
import sagemaker as sage
from sagemaker import get_execution_role, ModelPackage

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

sagemaker_session = sage.Session()
s3_bucket = sagemaker_session.default_bucket()
role = get_execution_role()

s3_client = sagemaker_session.boto_session.client("s3")
sm_runtime = boto3.client("sagemaker-runtime")

In [3]:
model_name = "JSL-Medical-DM-Small"

real_time_inference_instance_type = "ml.g5.2xlarge"
batch_transform_inference_instance_type = "ml.g5.2xlarge"

## 2. Create a deployable model from the model package.

In [ ]:
model = ModelPackage(
    role=role,
    model_package_arn=model_package_arn,
    sagemaker_session=sagemaker_session,
)

## Model Configuration Documentation

### Default Configuration
The container comes with the following default configurations:

| Parameter | Default Value | Description |
|---|---|---|
| **Maximum length of a row** | `16384` tokens | A row is the record, one question and its options. A longer row answers `422` |
| **Questions per request** | `64` | Questions of a decision request |
| **Options per question** | `255` | Options of a `choice` question or levels of a `score` question |
| **Documents per guard request** | `16384` | Documents of a guard request |
| **Documents per ensemble request** | `64` | Documents of an ensemble request |
| **Time limit of the decision model** | `60` seconds | Time limit of one request to the decision model |
| **Threads of each guard** | `4` | CPU threads of each guard |

### Hardcoded Settings
The following settings are hardcoded in the container and cannot be changed:

| Parameter | Value | Description |
|---|---|---|
| **`model`** | `Medical-DM-Small` | Name of the decision model in a request |
| **`model`** | `phi-guard-v0.7`, `phi-guard-v0.7-fast` | Names of the guards PHI Guard v0.7 and PHI Guard v0.7 Fast in a request |
| **`model`** | `phi-guard-ensemble` | Name of the ensemble in a request |
| **Endpoints** | `/ping`, `/invocations` | Health check and inference |

---

## Input Format

All requests are JSON (`application/json`). The fields of the body decide what runs:

| The body has | Runs | `model` |
|---|---|---|
| `state` and `questions` | the decision model | `Medical-DM-Small` |
| `docs` | one guard | `phi-guard-v0.7` (PHI Guard v0.7) or `phi-guard-v0.7-fast` (PHI Guard v0.7 Fast) |
| `docs` | the ensemble: a guard and the decision model, combined | `phi-guard-ensemble` |

Any other body answers `400` with the three accepted shapes.

### 1. Decision

`state` is the record (text, an object or a list). `questions` is an object of named questions, from 1 to 64 per request:

```json
{
    "model": "Medical-DM-Small",
    "state": "<the record>",
    "questions": {
        "question_1": {"type": "noul", "instructions": "<yes/no question>"},
        "question_2": {"type": "choice", "instructions": "<question>", "criteria": {"<option>": "<description>", "...": "..."}},
        "question_3": {"type": "score", "instructions": "<question>", "criteria": ["<lowest level>", "...", "<highest level>"]}
    }
}
```

A `noul` question returns the probability of yes (`criteria` with `true` and `false` descriptions is optional). A `choice` question returns the most likely option, a `confidence` and a probability for every option. A `score` question returns an expected level and a probability for every level.

### 2. Guard

`docs` is a list of documents, each with a unique `id` and a `text`, from 1 to 16,384 per request:

```json
{"model": "phi-guard-v0.7", "docs": [{"id": "note-001", "text": "<document>"}]}
```

### 3. Ensemble

The same body as a guard, with `"model": "phi-guard-ensemble"`, from 1 to 64 documents. Optional: `guard` (`phi-guard-v0.7` by default, or `phi-guard-v0.7-fast`), `aggregation` (`mean_logit` by default, or `mean`, `min`, `max`) and a `decision_score` (0 to 1) on a document. A document without `decision_score` is answered by the decision model of the container.

### Important Notes:
- **Model Name Requirement:** Always set `"model"` to one of the names above
- **Real-time Requests:** A real-time endpoint stops a request after 60 seconds. Send large requests as a batch transform job
- **Batch Transform:** Each input file is one request

## 3. Create a SageMaker Endpoint

If you want to understand how real-time inference with Amazon SageMaker works, see [Documentation](https://docs.aws.amazon.com/sagemaker/latest/dg/how-it-works-hosting.html).

The container answers `/ping` only after the model is loaded, so the timeouts of the download of the model files and of the startup health check are set to one hour.

In [ ]:
predictor = model.deploy(
    initial_instance_count=1,
    instance_type=real_time_inference_instance_type,
    endpoint_name=model_name,
    model_data_download_timeout=3600,
    container_startup_health_check_timeout=3600,
)

#### Initial setup

Three requests, one for every kind of request: a decision, a guard and the ensemble.

In [5]:
decision_request = {
    "model": "Medical-DM-Small",
    "state": {
        "age": 71,
        "sex": "female",
        "vitals": {
            "blood_pressure": "92/58",
            "heart_rate": 118,
            "temperature_c": 38.9,
            "oxygen_saturation": "91% on room air",
        },
        "labs": {
            "white_cell_count": "19.2 x10^9/L",
            "lactate": "4.1 mmol/L",
            "creatinine": "2.3 mg/dL (baseline 0.9)",
        },
        "medications": ["apixaban", "metformin", "lisinopril"],
        "complaint": "Two days of cough and fever, now confused.",
    },
    "questions": {
        "severity": {
            "type": "score",
            "instructions": "How worried should the care team be about this patient right now?",
            "criteria": [
                "Stable: no concern",
                "Mild concern: watch and repeat measurements",
                "Moderate concern: needs treatment on the ward",
                "Critical: needs intensive care assessment now",
            ],
        },
        "organ": {
            "type": "choice",
            "instructions": "Which problem is the most urgent?",
            "criteria": {
                "respiratory": "Lungs and breathing",
                "circulation": "Blood pressure and perfusion",
                "kidney": "Kidney function",
                "infection": "Infection and sepsis",
                "none": "Nothing urgent",
            },
        },
        "anticoagulated": {
            "type": "noul",
            "instructions": "Is the patient taking a blood thinner?",
        },
    },
}

CLEAN_NOTE = "Pt [NAME] seen on [DATE] for chest pain. Troponin negative x2, ECG normal sinus rhythm. Discharged home on aspirin 81 mg daily."
LEAKY_NOTE = (
    "DISCHARGE SUMMARY\n"
    "Patient Maria Gonzalez, a [AGE]-year-old [GENDER], was admitted to [HOSPITAL] on [DATE] with chest pain and shortness of breath. "
    "Troponin was negative twice. Started on Lipitor 40 mg daily and aspirin 81 mg daily. "
    "Echocardiogram showed an ejection fraction of 55 percent. Follow up with cardiology in two weeks. Contact [PHONE] with any questions."
)
docs = [{"id": "note-001", "text": CLEAN_NOTE}, {"id": "note-002", "text": LEAKY_NOTE}]

guard_request = {"model": "phi-guard-v0.7", "docs": docs}
ensemble_request = {"model": "phi-guard-ensemble", "docs": docs}

### 3.1 Real-time inference via Amazon SageMaker Endpoint

In [6]:
def invoke_realtime_endpoint(record):

    response = sm_runtime.invoke_endpoint(
        EndpointName=model_name,
        ContentType="application/json",
        Accept="application/json",
        Body=json.dumps(record),
    )

    return json.load(response["Body"])

In [7]:
def decision_table(result):
    rows = []
    for question, answer in result["answers"].items():
        if answer["type"] == "noul":
            rows.append({"question": question, "type": "noul", "answer": f'{answer["noul"]:.4f} (probability of yes)'})
        elif answer["type"] == "choice":
            rows.append({"question": question, "type": "choice", "answer": f'{answer["choice"]} (confidence {answer["confidence"]:.4f})'})
        else:
            rows.append({"question": question, "type": "score", "answer": f'{answer["score"]:.4f} (expected level)'})
    return pd.DataFrame(rows).set_index("question")


def guard_table(result):
    return pd.DataFrame(
        [
            {
                "id": r["id"],
                "leak score": r["score"],
                "leak": r["label"],
                "harmful score": r["harmful_score"],
                "exposure level": r["level"],
            }
            for r in result["results"]
        ]
    ).set_index("id")


def ensemble_table(result):
    return pd.DataFrame(
        [
            {
                "id": r["id"],
                "route": r["decision"],
                "combined score": r["score"],
                "guard": r["members"]["guard_score"],
                "decision model": r["members"]["decision_score"],
            }
            for r in result["results"]
        ]
    ).set_index("id")

#### Decision

In [8]:
result = invoke_realtime_endpoint(decision_request)
print(json.dumps(result, indent=2))

{
  "model": "Medical-DM-Small",
  "answers": {
    "severity": {
      "type": "score",
      "score": 2.9974,
      "probabilities": {
        "0": 0.0,
        "1": 0.0,
        "2": 0.0026,
        "3": 0.9974
      }
    },
    "organ": {
      "type": "choice",
      "choice": "infection",
      "confidence": 0.9957,
      "probabilities": {
        "respiratory": 0.001,
        "circulation": 0.0023,
        "kidney": 0.0002,
        "infection": 0.9965,
        "none": 0.0
      }
    },
    "anticoagulated": {
      "type": "noul",
      "noul": 0.9986
    }
  },
  "usage": {
    "input_tokens": 557,
    "output_tokens": 3
  }
}


In [9]:
decision_table(result)

,type,answer
question,,
severity,score,2.9974 (expected level)
organ,choice,infection (confidence 0.9957)
anticoagulated,noul,0.9986 (probability of yes)


#### Guard

In [10]:
result = invoke_realtime_endpoint(guard_request)
print(json.dumps(result, indent=2))

{
  "model": "phi-guard-v0.7",
  "default_cutoff": 0.5,
  "results": [
    {
      "id": "note-001",
      "score": 0.0101,
      "logit": -4.5816,
      "label": false,
      "cutoffs": {
        "0.5": false
      },
      "harmful_score": 0.0028,
      "harmful_logit": -5.8852,
      "harmful_label": false,
      "level": 0,
      "expected_level": 0.0002,
      "level_scores": [
        0.0001,
        0.0001,
        0.0,
        0.0,
        0.0,
        0.0
      ],
      "level_logits": [
        -9.3833,
        -9.3833,
        -13.0165,
        -16.7451,
        -17.2214,
        -17.2214
      ],
      "raw_score": 0.0,
      "raw_logit": -17.2178,
      "raw_label": false
    },
    {
      "id": "note-002",
      "score": 1.0,
      "logit": 13.5693,
      "label": true,
      "cutoffs": {
        "0.5": true
      },
      "harmful_score": 0.9995,
      "harmful_logit": 7.5929,
      "harmful_label": true,
      "level": 2,
      "expected_level": 1.7611,
      "level_sc

In [11]:
guard_table(result)

,leak score,leak,harmful score,exposure level
id,,,,
note-001,0.0101,False,0.0028,0
note-002,1.0000,True,0.9995,2


The same documents with PHI Guard v0.7 Fast:

In [12]:
result = invoke_realtime_endpoint({**guard_request, "model": "phi-guard-v0.7-fast"})
guard_table(result)

,leak score,leak,harmful score,exposure level
id,,,,
note-001,0.0205,False,0.0127,0
note-002,1.0000,True,0.9957,2


#### Ensemble

In [13]:
result = invoke_realtime_endpoint(ensemble_request)
print(json.dumps(result, indent=2))

{
  "model": "phi-guard-ensemble",
  "guard": "phi-guard-v0.7",
  "aggregation": "mean_logit",
  "results": [
    {
      "id": "note-001",
      "score": 0.001,
      "logit": -6.8959,
      "label": false,
      "harmful_score": 0.0005,
      "harmful_logit": -7.5477,
      "harmful_label": false,
      "decision": "clean",
      "agreement": true,
      "members": {
        "guard_score": 0.0101,
        "guard_harmful_score": 0.0028,
        "decision_score": 0.0
      },
      "level": 0,
      "level_scores": [
        0.0001,
        0.0001,
        0.0,
        0.0,
        0.0,
        0.0
      ],
      "raw_score": 0.0
    },
    {
      "id": "note-002",
      "score": 1.0,
      "logit": 11.0431,
      "label": true,
      "harmful_score": 0.9997,
      "harmful_logit": 8.0549,
      "harmful_label": true,
      "decision": "leak",
      "agreement": true,
      "members": {
        "guard_score": 1.0,
        "guard_harmful_score": 0.9995,
        "decision_score": 0.9998

In [14]:
ensemble_table(result)

,route,combined score,guard,decision model
id,,,,
note-001,clean,0.001,0.0101,0.0000
note-002,leak,1.000,1.0000,0.9998


Now that you have successfully performed a real-time inference, you do not need the endpoint any more. You can terminate the endpoint to avoid being charged.

In [ ]:
model.sagemaker_session.delete_endpoint(model_name)
model.sagemaker_session.delete_endpoint_config(model_name)

## 4. Batch inference

Each file in the input folder is one request. The three requests above are uploaded as three files.

In [17]:
batch_requests = {
    "input1.json": decision_request,
    "input2.json": guard_request,
    "input3.json": ensemble_request,
}
batch_input_path = f"s3://{s3_bucket}/{model_name}/validation-input/"
batch_output_path = f"s3://{s3_bucket}/{model_name}/validation-output/"

for file_name, request in batch_requests.items():
    s3_client.put_object(
        Bucket=s3_bucket,
        Key=f"{model_name}/validation-input/{file_name}",
        Body=json.dumps(request).encode("utf-8"),
    )

In [ ]:
transformer = model.transformer(
    instance_count=1,
    instance_type=batch_transform_inference_instance_type,
    strategy="SingleRecord",
    accept="application/json",
    output_path=batch_output_path,
)
transformer.transform(batch_input_path, content_type="application/json")
transformer.wait()

In [19]:
def retrieve_json_output_from_s3(file_name):
    parsed_url = urlparse(transformer.output_path)
    file_key = f"{parsed_url.path[1:]}{file_name}.out"
    response = s3_client.get_object(Bucket=s3_bucket, Key=file_key)
    return json.loads(response["Body"].read().decode("utf-8"))

#### Decision

In [20]:
result = retrieve_json_output_from_s3("input1.json")
print(json.dumps(result, indent=2))
decision_table(result)

{
  "model": "Medical-DM-Small",
  "answers": {
    "severity": {
      "type": "score",
      "score": 2.9973,
      "probabilities": {
        "0": 0.0,
        "1": 0.0,
        "2": 0.0027,
        "3": 0.9973
      }
    },
    "organ": {
      "type": "choice",
      "choice": "infection",
      "confidence": 0.9959,
      "probabilities": {
        "respiratory": 0.001,
        "circulation": 0.0021,
        "kidney": 0.0002,
        "infection": 0.9968,
        "none": 0.0
      }
    },
    "anticoagulated": {
      "type": "noul",
      "noul": 0.9986
    }
  },
  "usage": {
    "input_tokens": 557,
    "output_tokens": 3
  }
}


,type,answer
question,,
severity,score,2.9973 (expected level)
organ,choice,infection (confidence 0.9959)
anticoagulated,noul,0.9986 (probability of yes)


#### Guard

In [21]:
result = retrieve_json_output_from_s3("input2.json")
print(json.dumps(result, indent=2))
guard_table(result)

{
  "model": "phi-guard-v0.7",
  "default_cutoff": 0.5,
  "results": [
    {
      "id": "note-001",
      "score": 0.0101,
      "logit": -4.5816,
      "label": false,
      "cutoffs": {
        "0.5": false
      },
      "harmful_score": 0.0028,
      "harmful_logit": -5.8852,
      "harmful_label": false,
      "level": 0,
      "expected_level": 0.0002,
      "level_scores": [
        0.0001,
        0.0001,
        0.0,
        0.0,
        0.0,
        0.0
      ],
      "level_logits": [
        -9.3833,
        -9.3833,
        -13.0165,
        -16.7451,
        -17.2214,
        -17.2214
      ],
      "raw_score": 0.0,
      "raw_logit": -17.2178,
      "raw_label": false
    },
    {
      "id": "note-002",
      "score": 1.0,
      "logit": 13.5693,
      "label": true,
      "cutoffs": {
        "0.5": true
      },
      "harmful_score": 0.9995,
      "harmful_logit": 7.5929,
      "harmful_label": true,
      "level": 2,
      "expected_level": 1.7611,
      "level_sc

,leak score,leak,harmful score,exposure level
id,,,,
note-001,0.0101,False,0.0028,0
note-002,1.0000,True,0.9995,2


#### Ensemble

In [22]:
result = retrieve_json_output_from_s3("input3.json")
print(json.dumps(result, indent=2))
ensemble_table(result)

{
  "model": "phi-guard-ensemble",
  "guard": "phi-guard-v0.7",
  "aggregation": "mean_logit",
  "results": [
    {
      "id": "note-001",
      "score": 0.001,
      "logit": -6.8959,
      "label": false,
      "harmful_score": 0.0005,
      "harmful_logit": -7.5477,
      "harmful_label": false,
      "decision": "clean",
      "agreement": true,
      "members": {
        "guard_score": 0.0101,
        "guard_harmful_score": 0.0028,
        "decision_score": 0.0
      },
      "level": 0,
      "level_scores": [
        0.0001,
        0.0001,
        0.0,
        0.0,
        0.0,
        0.0
      ],
      "raw_score": 0.0
    },
    {
      "id": "note-002",
      "score": 1.0,
      "logit": 10.8404,
      "label": true,
      "harmful_score": 0.9996,
      "harmful_logit": 7.8522,
      "harmful_label": true,
      "decision": "leak",
      "agreement": true,
      "members": {
        "guard_score": 1.0,
        "guard_harmful_score": 0.9995,
        "decision_score": 0.9997

,route,combined score,guard,decision model
id,,,,
note-001,clean,0.001,0.0101,0.0000
note-002,leak,1.000,1.0000,0.9997


Congratulations! You just verified that the batch transform job is working as expected. Since the model is not required, you can delete it. Note that you are deleting the deployable model. Not the model package.

In [24]:
model.delete_model()

INFO:sagemaker:Deleting model with name: JSL-Medical-DM-Small-2026-11-20-20-55-57-967


### Unsubscribe to the listing (optional)

If you would like to unsubscribe to the model package, follow these steps. Before you cancel the subscription, ensure that you do not have any [deployable model](https://console.aws.amazon.com/sagemaker/home#/models) created from the model package or using the algorithm. Note - You can find this information by looking at the container name associated with the model.

**Steps to unsubscribe to product from AWS Marketplace**:
1. Navigate to __Machine Learning__ tab on [__Your Software subscriptions page__](https://aws.amazon.com/marketplace/ai/library?productType=ml&ref_=mlmp_gitdemo_indust)
2. Locate the listing that you want to cancel the subscription for, and then choose __Cancel Subscription__  to cancel the subscription.